# KDFND revised experiments: leakage audit, leak-resistant protocols, ablations, statistics, efficiency

This notebook replaces the two earlier notebooks (`new-kurdishkdfnd` and `new-kdfnd-without-source`).
It answers the reviewers' requests point by point and **writes every number used in the paper to CSV/JSON files in `OUT_DIR`**
so nothing has to be copied by hand from console output.

| Section | Reviewer point |
|---|---|
| 1 Setup + hardware log | R1-10, R1-11 |
| 2 Load, label encoding (explicit), counts | R1-9 |
| 3 Data audit: duplicates, source-label purity, length, dates, source names inside text | R1-2, R1-4, R1-5 |
| 4 Features, models, metrics | R1-9, R1-11 |
| 5 Protocols: random, de-duplicated, source-disjoint, temporal | R1-3 |
| 6 Ablation: lookup / source-only / length-only / content-only / combined | R1-6 |
| 7 Full 8-model suite (tuned once, then repeated with fixed hyper-parameters) | R1-7, R3-4 |
| 8 Deep models (CNN, BiLSTM-Attention) with real timing | R1-10 |
| 9 Statistics: mean, SD, 95% CI, corrected resampled t-test, Wilcoxon, Holm | R1-7 |
| 10 Efficiency: fit time, latency, size, memory | R1-10 |
| 11 Convergence evidence | R3-3 |
| 12 Export | all |

**Before running:** on Kaggle turn on a GPU accelerator (the earlier runs show `cuInit failed`, i.e. everything ran on CPU).
Set `QUICK = True` first to check that the whole notebook runs end-to-end in a few minutes, then set `QUICK = False`.
Expected full-run cost (4-core CPU + one GPU): roughly 4-8 hours, dominated by RF/XGBoost/Stacking. If that is too long,
lower `N_SEEDS_HEAVY`, `N_SEEDS_DL`, or set `RUN_TUNE = False` (then report the fixed hyper-parameters as such).

In [ ]:
# ---------------------------------------------------------------- 1. SETUP
import os, gc, re, json, time, string, platform, subprocess, warnings, itertools, math, sys
from urllib.parse import urlparse
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             matthews_corrcoef, cohen_kappa_score, brier_score_loss, mutual_info_score,
                             confusion_matrix)
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False
    print('xgboost not available -> XGBoost rows will be skipped')
try:
    import tensorflow as tf
    HAVE_TF = True
except Exception:
    HAVE_TF = False
    print('tensorflow not available -> deep models will be skipped')
import joblib, psutil

# ----------------------------- CONFIGURATION (edit here only)
QUICK     = False        # True = tiny smoke test (subsample, 2 seeds, no grid search, 2 DL epochs)
RUN_DL    = True         # CNN + BiLSTM-Attention
RUN_TUNE  = True         # grid/random search once on the primary random split (as in the original paper)
DATA_PATH = '/kaggle/input/datasets/azadmk/kurdishkdfnd/KDFND_Anlyzed_Cleaned_Filtered_Labeld.xlsx'
OUT_DIR   = '/kaggle/working/outputs'
BASE_SEED = 42
N_SEEDS   = 10           # repeated random splits for fast models
N_SEEDS_DL = 3           # repeated runs for deep models (each BiLSTM run is expensive on CPU)
N_SEEDS_HEAVY = 3        # RF / XGBoost / Voting / Stacking are repeated on this many random splits (LR/SVM use N_SEEDS)
DL_PROTOCOLS = ['random', 'source_disjoint', 'temporal']   # add 'random_dedup' if time allows
FORCE_BILSTM_ON_CPU = False   # BiLSTM needs a GPU (about 4 min/epoch on CPU in the earlier runs)
TEST_FRAC = 0.2
if QUICK:
    N_SEEDS, N_SEEDS_DL, N_SEEDS_HEAVY, RUN_TUNE = 2, 1, 1, False
os.makedirs(OUT_DIR, exist_ok=True)
SEEDS    = [BASE_SEED + i for i in range(N_SEEDS)]
DL_SEEDS = [BASE_SEED + i for i in range(N_SEEDS_DL)]
np.random.seed(BASE_SEED)
os.environ['PYTHONHASHSEED'] = str(BASE_SEED)
if HAVE_TF:
    tf.keras.utils.set_random_seed(BASE_SEED)

def save_json(obj, name):
    with open(os.path.join(OUT_DIR, name), 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def hardware_report():
    hw = {'platform': platform.platform(), 'python': sys.version.split()[0],
          'cpu_logical_cores': os.cpu_count(), 'ram_gb': round(psutil.virtual_memory().total / 2**30, 1)}
    try:
        with open('/proc/cpuinfo') as f:
            hw['cpu_model'] = next(l.split(':')[1].strip() for l in f if l.startswith('model name'))
    except Exception:
        hw['cpu_model'] = platform.processor()
    import sklearn, scipy
    hw['versions'] = {'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
                      'scipy': scipy.__version__}
    if HAVE_XGB:
        import xgboost; hw['versions']['xgboost'] = xgboost.__version__
    if HAVE_TF:
        hw['versions']['tensorflow'] = tf.__version__
        hw['gpus_visible_to_tensorflow'] = [d.name for d in tf.config.list_physical_devices('GPU')]
    try:
        hw['nvidia_smi'] = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                          capture_output=True, text=True, timeout=10).stdout.strip() or 'none'
    except Exception:
        hw['nvidia_smi'] = 'none'
    return hw

HW = hardware_report()
save_json(HW, 'hardware.json')
print(json.dumps(HW, indent=2))
if HAVE_TF and not HW.get('gpus_visible_to_tensorflow'):
    print('\nWARNING: TensorFlow sees no GPU. Deep-model timings below are CPU timings and must be reported as such.')

In [ ]:
# ---------------------------------------------------------------- 2. LOAD, CLEAN, LABELS
# NOTE (bug in the earlier notebooks): LabelEncoder sorts classes alphabetically (Fake=0, Real=1) although the
# comments said Real=0, Fake=1, so confusion-matrix axes and "false positive/negative" wording were swapped.
# Here the mapping is explicit: Fake = 1 = POSITIVE class.
if 'df_input' in globals():                 # (used only by the smoke test)
    df = df_input.copy()
else:
    df = pd.read_excel(DATA_PATH)
n_raw = len(df)
df = df.dropna(subset=['Text']).reset_index(drop=True)
n_after_missing = len(df)
assert set(df['label'].unique()) <= {'Real', 'Fake'}, df['label'].unique()
df['y'] = (df['label'] == 'Fake').astype(int)          # Fake = 1 (positive), Real = 0

URL_RE = re.compile(r'http\S+|www\S+')
def clean_text(t):
    t = URL_RE.sub('', str(t))
    t = re.sub(r'@\w+|#', '', t)
    t = t.translate(str.maketrans('', '', string.punctuation))
    t = re.sub(r'\d+', '', t)
    return ' '.join(t.split())
# Kurdish has no letter case, so "lower-casing" only affects the Latin source names; the TF-IDF vectoriser and the
# Keras tokenizer both lower-case internally, which is what the paper must say (not a separate preprocessing step).
df['cleaned_text'] = df['Text'].apply(clean_text)
df['source'] = df['Source'].astype(str).str.strip().str.lower().replace({'nan': 'unknown', '': 'unknown'})
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
df['domain'] = df['URL'].astype(str).apply(lambda u: urlparse(u).netloc.lower().replace('www.', '') if u.startswith('http') else 'none')
df['src_token'] = 'src_' + df['source'].str.replace(r'\W+', '_', regex=True)

if QUICK and len(df) > 12000:
    df = df.sample(12000, random_state=BASE_SEED).reset_index(drop=True)
    print('QUICK mode: subsampled to 12,000 rows')

counts = {'n_raw_rows': int(n_raw), 'n_missing_text_dropped': int(n_raw - n_after_missing),
          'n_after_dropping_missing_text': int(len(df)),
          'n_real': int((df.y == 0).sum()), 'n_fake': int((df.y == 1).sum()),
          'positive_class': 'Fake (=1)'}
print(counts)
save_json(counts, 'counts.json')

In [ ]:
# ---------------------------------------------------------------- 3. DATA AUDIT
audit = {}

# 3.1 duplicates ---------------------------------------------------------------------------------
dup_mask = df.duplicated('cleaned_text', keep=False)
audit['rows_with_exact_duplicate_cleaned_text'] = int(dup_mask.sum())
audit['redundant_rows_if_deduplicated'] = int(df.duplicated('cleaned_text', keep='first').sum())
g = df.groupby('cleaned_text')['y'].nunique()
audit['texts_appearing_with_both_labels'] = int((g > 1).sum())
audit['rows_empty_after_cleaning'] = int((df.cleaned_text.str.len() == 0).sum())
rdup = df.duplicated('Text', keep=False)
audit['rows_with_exact_duplicate_raw_text'] = int(rdup.sum())

def near_duplicate_rate(train_txt, test_txt, thr=0.9, max_test=5000, seed=BASE_SEED):
    """share of test documents whose nearest training document has TF-IDF cosine >= thr, and label agreement"""
    rng = np.random.RandomState(seed)
    ti = rng.choice(len(test_txt), min(max_test, len(test_txt)), replace=False)
    vec = TfidfVectorizer(max_df=0.5, min_df=2, dtype=np.float32)
    Xtr = vec.fit_transform(train_txt); Xte = vec.transform(np.asarray(test_txt)[ti])
    best, arg = np.zeros(Xte.shape[0]), np.zeros(Xte.shape[0], dtype=int)
    XtrT = Xtr.T.tocsc()
    for s in range(0, Xte.shape[0], 500):
        S = (Xte[s:s + 500] @ XtrT)
        best[s:s + 500] = np.asarray(S.max(axis=1).todense()).ravel()
        arg[s:s + 500] = np.asarray(S.argmax(axis=1)).ravel()
    return ti, best, arg

tr_i, te_i = train_test_split(np.arange(len(df)), test_size=TEST_FRAC, random_state=BASE_SEED, stratify=df.y)
ti, best, arg = near_duplicate_rate(df.cleaned_text.values[tr_i], df.cleaned_text.values[te_i])
for thr in (0.8, 0.9, 0.95, 0.99):
    audit[f'test_docs_with_train_neighbour_cosine>={thr}'] = float((best >= thr).mean())
close = best >= 0.9
audit['label_agreement_among_neighbours>=0.9'] = float((df.y.values[te_i][ti][close] == df.y.values[tr_i][arg][close]).mean()) if close.any() else None
print('3.1 duplicates:', {k: v for k, v in audit.items() if 'dup' in k or 'neigh' in k or 'both' in k})

# 3.2 source-label relationship ------------------------------------------------------------------
ct = pd.crosstab(df.source, df.y).rename(columns={0: 'real', 1: 'fake'})
ct['n'] = ct.real + ct.fake
ct['fake_rate'] = ct.fake / ct.n
ct = ct.sort_values('n', ascending=False)
ct.to_csv(os.path.join(OUT_DIR, 'source_label_table.csv'))
audit['n_unique_sources'] = int(len(ct))
audit['sources_with_both_labels'] = int(((ct.real > 0) & (ct.fake > 0)).sum())
audit['sources_fake_rate<=0.01_or>=0.99'] = int(((ct.fake_rate <= 0.01) | (ct.fake_rate >= 0.99)).sum())
audit['share_of_articles_in_pure_sources(>=99%)'] = float(ct.loc[(ct.fake_rate <= 0.01) | (ct.fake_rate >= 0.99), 'n'].sum() / len(df))
audit['accuracy_of_in-sample_majority-label-per-source'] = float(ct[['real', 'fake']].max(axis=1).sum() / len(df))
audit['mutual_information_source_label_bits'] = float(mutual_info_score(df.source, df.y) / np.log(2))
audit['label_entropy_bits'] = float(-sum(p * np.log2(p) for p in df.y.value_counts(normalize=True)))
chi2 = stats.chi2_contingency(ct[['real', 'fake']].values)[0]
audit['cramers_v_source_label'] = float(np.sqrt(chi2 / (len(df) * 1)))
ctd = pd.crosstab(df.domain, df.y)
audit['accuracy_of_in-sample_majority-label-per-url-domain'] = float(ctd.max(axis=1).sum() / len(df))
print('3.2 source vs label:', {k: v for k, v in audit.items() if 'source' in k or 'domain' in k})
print(ct.head(25).to_string())

fig, ax = plt.subplots(figsize=(8, 7))
top = ct.head(20).iloc[::-1]
ax.barh(top.index, top.real, color='tab:green', label='Real'); ax.barh(top.index, top.fake, left=top.real, color='tab:red', label='Fake')
ax.set_xlabel('Articles'); ax.set_title('Top-20 sources: label composition'); ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_source_label_composition.png'), dpi=300); plt.show()

# 3.3 length / style -----------------------------------------------------------------------------
df['char_count_raw'] = df.Text.astype(str).str.len()
df['word_count_raw'] = df.Text.astype(str).str.split().str.len()
for lab, nm in ((0, 'real'), (1, 'fake')):
    s = df[df.y == lab]
    audit[f'mean_chars_{nm}'] = float(s.char_count_raw.mean()); audit[f'mean_words_{nm}'] = float(s.word_count_raw.mean())
    audit[f'median_words_{nm}'] = float(s.word_count_raw.median())
audit['auc_word_count_alone'] = float(roc_auc_score(df.y, df.word_count_raw))
audit['auc_char_count_alone'] = float(roc_auc_score(df.y, df.char_count_raw))
print('3.3 length:', {k: round(v, 3) for k, v in audit.items() if 'chars' in k or 'words' in k or 'auc_' in k})

# 3.4 dates --------------------------------------------------------------------------------------
audit['date_parse_failures'] = int(df.date_parsed.isna().sum())
if df.date_parsed.notna().any():
    audit['date_min'] = str(df.date_parsed.min()); audit['date_max'] = str(df.date_parsed.max())
    m = df.dropna(subset=['date_parsed']).groupby(df.date_parsed.dt.to_period('M')).y.agg(['mean', 'count'])
    m.to_csv(os.path.join(OUT_DIR, 'monthly_fake_rate.csv'))
    print('3.4 dates:', audit['date_min'], '->', audit['date_max']); print(m.round(3).T.to_string())

# 3.5 does the content-only text still carry the source? ------------------------------------------
def contains_source_name(row):
    toks = [t for t in re.split(r'\W+', row.source) if len(t) >= 3]
    return any(t in str(row.Text).lower() for t in toks) if toks else False
sample = df.sample(min(20000, len(df)), random_state=BASE_SEED)
audit['share_of_articles_whose_text_contains_latin_source_name'] = float(sample.apply(contains_source_name, axis=1).mean())
vec = TfidfVectorizer(max_features=10000, min_df=2, max_df=0.8)
Xc = vec.fit_transform(df.cleaned_text)
lr = LogisticRegression(C=1.0, solver='liblinear', max_iter=2000).fit(Xc, df.y)
fn = np.array(vec.get_feature_names_out()); o = np.argsort(lr.coef_[0])
print('3.5 content-only tokens most indicative of REAL:', list(fn[o[:40]]))
print('    content-only tokens most indicative of FAKE:', list(fn[o[-40:]][::-1]))
audit['top40_tokens_real'] = list(fn[o[:40]]); audit['top40_tokens_fake'] = list(fn[o[-40:]][::-1])
del Xc, lr; gc.collect()

save_json(audit, 'audit_summary.json')

In [ ]:
# ---------------------------------------------------------------- 4. FEATURES, MODELS, METRICS
KU_STOP = ["ئێمە","ئێوە","ئەم","ئەو","ئەوان","ئەوەی","بۆ","بێ","بێجگە","بە","بەبێ","بەدەم","بەردەم","بەرلە","بەرەوی","بەرەوە","بەلای",
           "بەپێی","تۆ","تێ","جگە","دوای","دوو","دە","دەکات","دەگەڵ","سەر","لێ","لە","لەبابەت","لەباتی","لەبارەی","لەبرێتی","لەبن",
           "لەبەر","لەبەینی","لەدەم","لەرێ","لەرێگا","لەرەوی","لەسەر","لەلایەن","لەناو","لەنێو","لەو","لەپێناوی","لەژێر","لەگەڵ",
           "من","ناو","نێوان","هەر","هەروەها","و","وەک","پاش","پێ","پێش","چەند","کرد","کە","ی"]   # list used in the original study
KU_STOP_SET = set(KU_STOP)

# Hand-crafted length/style features, computed on the RAW text (the earlier code computed punctuation, digit and
# upper-case counts on text from which those characters had already been removed, so they were constant).
# Upper-case features are dropped: Kurdish (Arabic script) has no letter case.
def style_features(texts):
    rows = []
    for t in texts:
        t = str(t); w = t.split(); nw = max(len(w), 1); nc = max(len(t), 1)
        letters = [c for c in t if c.isalpha()]
        latin = sum(1 for c in letters if 'a' <= c.lower() <= 'z')
        sw = sum(1 for x in w if x in KU_STOP_SET)
        pun = sum(1 for c in t if c in string.punctuation or c in '،؛؟«»…')
        dig = sum(1 for c in t if c.isdigit())
        rows.append([len(w), len(t), len(set(w)), len(set(w)) / nw, sum(len(x) for x in w) / nw, sw, sw / nw,
                     pun, pun / nc, dig, dig / nc, latin / max(len(letters), 1), t.count('\n'), len(URL_RE.findall(t))])
    return np.array(rows, dtype=float)
STYLE_NAMES = ['word_count', 'char_count', 'unique_words', 'unique_ratio', 'avg_word_len', 'stopword_count', 'stopword_ratio',
               'punct_count', 'punct_ratio', 'digit_count', 'digit_ratio', 'latin_letter_ratio', 'newlines', 'urls']
t0 = time.time(); STYLE = style_features(df.Text.values); print('style features', STYLE.shape, f'{time.time()-t0:.1f}s')

TFIDF_KW = dict(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.8, stop_words=KU_STOP)

def make_preprocessor(rep):
    """rep in content | source | combined | length | content+length"""
    tf_ = ('text', TfidfVectorizer(**TFIDF_KW), 'cleaned_text')
    sr_ = ('src', OneHotEncoder(handle_unknown='ignore'), ['source'])
    ln_ = ('len', Pipeline([('sc', StandardScaler())]), STYLE_NAMES)
    parts = {'content': [tf_], 'source': [sr_], 'combined': [tf_, sr_], 'length': [ln_], 'content+length': [tf_, ln_]}[rep]
    return ColumnTransformer(parts, sparse_threshold=0.3)

FEATURES = df[['cleaned_text', 'source']].copy()
for j, n in enumerate(STYLE_NAMES):
    FEATURES[n] = STYLE[:, j]

class SourceLookup:
    """Publisher look-up baseline: predict the majority training label of the article's source; unseen source -> global majority"""
    def fit(self, X, y):
        d = pd.DataFrame({'s': X['source'].values, 'y': np.asarray(y)})
        self.map_ = d.groupby('s').y.mean().to_dict(); self.glob_ = float(np.mean(y)); return self
    def decision_function(self, X):
        return X['source'].map(self.map_).fillna(self.glob_).values
    def predict(self, X):
        return (self.decision_function(X) >= 0.5).astype(int)
    def predict_proba(self, X):
        p = self.decision_function(X); return np.c_[1 - p, p]

def base_models(seed):
    m = {'LR': LogisticRegression(C=1.0, solver='liblinear', max_iter=2000, random_state=seed),
         'SVM': LinearSVC(C=1.0, max_iter=20000, random_state=seed),
         'RF': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed)}
    if HAVE_XGB:
        m['XGB'] = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, tree_method='hist',
                                 n_jobs=-1, eval_metric='logloss', random_state=seed)
    return m

def ensembles(seed):
    b = base_models(seed); est = [(k, b[k]) for k in ('LR', 'RF', 'XGB') if k in b]
    return {'Voting': VotingClassifier(est, voting='soft'),
            'Stacking': StackingClassifier(est, final_estimator=LogisticRegression(C=0.5, random_state=seed), cv=3, stack_method='predict_proba')}

def get_scores(model, X):
    if hasattr(model, 'predict_proba'): return model.predict_proba(X)[:, 1]
    return model.decision_function(X)          # LinearSVC: rank score is enough for ROC-AUC

def metrics(y, pred, score=None):
    d = {'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
         'recall': recall_score(y, pred, zero_division=0), 'f1': f1_score(y, pred, zero_division=0),
         'macro_f1': f1_score(y, pred, average='macro'), 'mcc': matthews_corrcoef(y, pred), 'kappa': cohen_kappa_score(y, pred)}
    d['auc'] = roc_auc_score(y, score) if (score is not None and len(np.unique(y)) == 2) else np.nan
    tn, fp, fn_, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    d.update({'tn': tn, 'fp': fp, 'fn': fn_, 'tp': tp})
    return d

def boot_ci(y, pred, n=1000, seed=BASE_SEED):
    rng = np.random.RandomState(seed); y = np.asarray(y); pred = np.asarray(pred); v = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y)); v.append(f1_score(y[i], pred[i], zero_division=0))
    return np.percentile(v, [2.5, 97.5])

In [ ]:
# ---------------------------------------------------------------- 5. PROTOCOLS
def split_random(seed):
    return train_test_split(np.arange(len(df)), test_size=TEST_FRAC, random_state=seed, stratify=df.y)

_dedup_keep = ~df.duplicated('cleaned_text', keep='first')                                                # keep one copy of each text
_dedup_keep &= ~df.cleaned_text.isin(g[g > 1].index) & (df.cleaned_text.str.len() > 0)                   # drop texts with conflicting labels / empty
DEDUP_IDX = np.where(_dedup_keep)[0]
def split_dedup(seed):
    tr, te = train_test_split(DEDUP_IDX, test_size=TEST_FRAC, random_state=seed, stratify=df.y.values[DEDUP_IDX]); return tr, te

def split_source_disjoint(seed):
    """no source appears in both train and test (StratifiedGroupKFold, first fold = ~20 % test)"""
    sgkf = StratifiedGroupKFold(n_splits=int(round(1 / TEST_FRAC)), shuffle=True, random_state=seed)
    tr, te = next(sgkf.split(np.zeros(len(df)), df.y, groups=df.source))
    assert not set(df.source.values[tr]) & set(df.source.values[te])
    return tr, te

_dated = np.where(df.date_parsed.notna())[0]
_dated = _dated[np.argsort(df.date_parsed.values[_dated], kind='stable')]
def split_temporal(fold):
    """rolling origin: split the dated articles into 6 chronological blocks; train on the first `fold` blocks, test on the next one (fold=2..5)
    fold=5 is the classical 'train on the earliest 5/6, test on the latest 1/6'"""
    blocks = np.array_split(_dated, 6)
    return np.concatenate(blocks[:fold]), blocks[fold]

PROTOCOLS = {'random': split_random, 'random_dedup': split_dedup, 'source_disjoint': split_source_disjoint}
if len(_dated) > 1000:
    PROTOCOLS['temporal'] = split_temporal
print({k: 'ok' for k in PROTOCOLS})

def protocol_runs(proto, n_seeds):
    if proto == 'temporal': return [('fold%d' % f, f) for f in range(2, 6)]
    return [('seed%d' % s, s) for s in SEEDS[:n_seeds]]

proto_info = []
for proto, fn_ in PROTOCOLS.items():
    for tag, arg_ in protocol_runs(proto, 1):
        tr, te = fn_(arg_)
        proto_info.append({'protocol': proto, 'run': tag, 'n_train': len(tr), 'n_test': len(te),
                           'test_fake_share': float(df.y.values[te].mean()),
                           'sources_in_test': int(len(set(df.source.values[te]))),
                           'test_sources_unseen_in_train': int(len(set(df.source.values[te]) - set(df.source.values[tr]))),
                           'test_rows_from_unseen_sources': float((~pd.Series(df.source.values[te]).isin(set(df.source.values[tr]))).mean())})
proto_info = pd.DataFrame(proto_info); proto_info.to_csv(os.path.join(OUT_DIR, 'protocol_info.csv'), index=False)
print(proto_info.to_string())

In [ ]:
# ---------------------------------------------------------------- 6/7. ABLATION + MODEL SUITE (classical models, repeated runs)
RESULTS = []          # long format, one row per (protocol, run, representation, model)
def run_one(proto, tag, arg_, rep, name, model, tr, te):
    Xtr, Xte = FEATURES.iloc[tr], FEATURES.iloc[te]; ytr, yte = df.y.values[tr], df.y.values[te]
    if name in ('SourceLookup', 'Majority'): pipe = model
    else: pipe = Pipeline([('prep', make_preprocessor(rep)), ('clf', model)])
    t0 = time.time(); pipe.fit(Xtr, ytr); fit_s = time.time() - t0
    t0 = time.time(); pred = pipe.predict(Xte); pred_s = time.time() - t0
    score = get_scores(pipe, Xte)
    row = {'protocol': proto, 'run': tag, 'representation': rep, 'model': name, 'fit_seconds': fit_s,
           'predict_seconds_test_set': pred_s, 'n_test': len(te)}
    row.update(metrics(yte, pred, score)); RESULTS.append(row); return row

FAST = ['LR', 'SVM']                                       # run for every representation and every protocol
HEAVY = ['RF', 'XGB', 'Voting', 'Stacking']                # run for content and combined representations
if QUICK: HEAVY = ['XGB'] if HAVE_XGB else ['RF']

t_all = time.time()
for proto, fn_ in PROTOCOLS.items():
    for tag, arg_ in protocol_runs(proto, N_SEEDS):
        tr, te = fn_(arg_)
        # trivial baselines
        run_one(proto, tag, arg_, 'none', 'Majority', DummyClassifier(strategy='most_frequent'), tr, te)
        run_one(proto, tag, arg_, 'source', 'SourceLookup', SourceLookup(), tr, te)
        for rep in ('source', 'length', 'content', 'combined', 'content+length'):
            for nm in FAST:
                run_one(proto, tag, arg_, rep, nm, base_models(BASE_SEED)[nm], tr, te)
        run_idx = int(tag.replace('seed', '').replace('fold', '')) - (BASE_SEED if tag.startswith('seed') else 2)
        heavy_here = (proto == 'temporal') or (run_idx < N_SEEDS_HEAVY)
        for rep in ('length', 'content', 'combined'):
            for nm in (HEAVY if heavy_here else []):
                if rep == 'length' and nm in ('Voting', 'Stacking'): continue
                mdl = {**base_models(BASE_SEED), **ensembles(BASE_SEED)}.get(nm)
                if mdl is None: continue
                run_one(proto, tag, arg_, rep, nm, mdl, tr, te)
        pd.DataFrame(RESULTS).to_csv(os.path.join(OUT_DIR, 'results_long_classical.csv'), index=False)
        print(f'[{time.time()-t_all:7.0f}s] done {proto}/{tag}')
res = pd.DataFrame(RESULTS)

In [ ]:
# ---------------------------------------------------------------- 7b. TUNED SUITE ON THE PRIMARY SPLIT (as in the original paper) + best params
TUNED = []; BEST_PARAMS = {}
if RUN_TUNE:
    tr, te = split_random(BASE_SEED)
    grids = {'LR': (GridSearchCV, LogisticRegression(random_state=BASE_SEED, solver='liblinear', max_iter=2000), {'clf__C': [0.1, 0.5, 1.0, 10.0]}),
             'SVM': (GridSearchCV, LinearSVC(random_state=BASE_SEED, max_iter=20000), {'clf__C': [0.1, 1.0, 10.0], 'clf__loss': ['hinge', 'squared_hinge']}),
             'RF': (RandomizedSearchCV, RandomForestClassifier(random_state=BASE_SEED, n_jobs=-1),
                    {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20, None], 'clf__min_samples_split': [2, 5, 10], 'clf__min_samples_leaf': [1, 2, 4]})}
    if HAVE_XGB:
        grids['XGB'] = (RandomizedSearchCV, XGBClassifier(random_state=BASE_SEED, tree_method='hist', n_jobs=-1, eval_metric='logloss'),
                        {'clf__n_estimators': [100, 200], 'clf__max_depth': [4, 6, 8], 'clf__learning_rate': [0.01, 0.05, 0.1], 'clf__subsample': [0.8, 1.0]})
    for rep in ('combined', 'content'):
        for nm, (S, est, grid) in grids.items():
            pipe = Pipeline([('prep', make_preprocessor(rep)), ('clf', est)])
            kw = dict(n_iter=10, random_state=BASE_SEED) if S is RandomizedSearchCV else {}
            search = S(pipe, grid, cv=3, scoring='f1', n_jobs=1, **kw)       # n_jobs=1 here; the estimators themselves are parallel
            t0 = time.time(); search.fit(FEATURES.iloc[tr], df.y.values[tr]); tune_s = time.time() - t0
            pred = search.predict(FEATURES.iloc[te]); sc = get_scores(search, FEATURES.iloc[te])
            r = {'representation': rep, 'model': nm, 'tuning_plus_refit_seconds': tune_s}; r.update(metrics(df.y.values[te], pred, sc)); r['f1_ci_low'], r['f1_ci_high'] = boot_ci(df.y.values[te], pred)
            TUNED.append(r); BEST_PARAMS[f'{rep}/{nm}'] = search.best_params_
            print(rep, nm, {k: round(v, 4) for k, v in r.items() if k in ('f1', 'auc')}, search.best_params_)
    pd.DataFrame(TUNED).to_csv(os.path.join(OUT_DIR, 'results_tuned_primary_split.csv'), index=False)
    save_json(BEST_PARAMS, 'best_params.json')

In [ ]:
# ---------------------------------------------------------------- 8. DEEP MODELS (real timing, explicit seeds)
DL_ROWS, DL_HIST = [], {}
if HAVE_TF and RUN_DL:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras import layers as L, Model, Input
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

    MAXLEN, VOCAB, EMB = 150, 20000, 128
    EPOCHS = {'CNN': 20, 'BiLSTM-Attention': 15}
    if QUICK: EPOCHS = {'CNN': 2, 'BiLSTM-Attention': 2}

    class EpochTimer(Callback):
        def on_train_begin(self, logs=None): self.t = []
        def on_epoch_begin(self, epoch, logs=None): self._s = time.time()
        def on_epoch_end(self, epoch, logs=None): self.t.append(time.time() - self._s)

    def build_cnn():
        m = tf.keras.Sequential([Input(shape=(MAXLEN,)), L.Embedding(VOCAB, EMB), L.Conv1D(128, 3, activation='relu'), L.MaxPooling1D(2),
                                 L.Conv1D(128, 3, activation='relu'), L.GlobalMaxPooling1D(), L.Dropout(0.5), L.Dense(64, activation='relu'),
                                 L.Dropout(0.3), L.Dense(1, activation='sigmoid')])
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy']); return m

    def build_bilstm():
        i = Input(shape=(MAXLEN,)); x = L.Embedding(VOCAB, EMB)(i)
        x = L.Bidirectional(L.LSTM(64, return_sequences=True))(x); x = L.Dropout(0.3)(x)
        a = L.Dense(1, activation='tanh')(x); a = L.Flatten()(a); a = L.Activation('softmax')(a)
        a = L.RepeatVector(128)(a); a = L.Permute([2, 1])(a); x = L.Multiply()([x, a]); x = L.GlobalAveragePooling1D()(x)
        x = L.Dense(64, activation='relu')(x); x = L.Dropout(0.5)(x); o = L.Dense(1, activation='sigmoid')(x)
        m = Model(i, o); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy']); return m
    BUILD = {'CNN': build_cnn, 'BiLSTM-Attention': build_bilstm}
    HAS_GPU = bool(HW.get('gpus_visible_to_tensorflow'))
    DL_MODELS = list(BUILD) if (HAS_GPU or FORCE_BILSTM_ON_CPU or QUICK) else ['CNN']
    print('Deep models to run:', DL_MODELS, '| GPU visible:', HAS_GPU)

    def dl_text(idx, rep):
        t = df.cleaned_text.values[idx]
        return (df.src_token.values[idx] + ' ' + t) if rep == 'combined' else t

    def run_dl(proto, tag, arg_, seed, rep, name, tr, te):
        tf.keras.utils.set_random_seed(seed)
        tr_in, va_in = train_test_split(tr, test_size=0.2, random_state=seed, stratify=df.y.values[tr])   # shuffled validation split
        tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>'); tok.fit_on_texts(dl_text(tr_in, rep))      # tokenizer fitted on training portion only
        pad = lambda idx: pad_sequences(tok.texts_to_sequences(dl_text(idx, rep)), maxlen=MAXLEN, padding='post', truncating='post')
        Xtr, Xva, Xte = pad(tr_in), pad(va_in), pad(te); ytr, yva, yte = df.y.values[tr_in], df.y.values[va_in], df.y.values[te]
        m = BUILD[name](); timer = EpochTimer()
        cbs = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6), timer]
        rss0 = psutil.Process().memory_info().rss
        t0 = time.time(); h = m.fit(Xtr, ytr, epochs=EPOCHS[name], batch_size=64, validation_data=(Xva, yva), callbacks=cbs, verbose=0); fit_s = time.time() - t0
        t0 = time.time(); p = m.predict(Xte, batch_size=256, verbose=0).ravel(); pred_s = time.time() - t0
        row = {'protocol': proto, 'run': tag, 'representation': rep, 'model': name, 'seed': seed, 'fit_seconds': fit_s, 'epochs_run': len(h.epoch),
               'seconds_per_epoch': float(np.mean(timer.t)), 'best_epoch': int(np.argmin(h.history['val_loss']) + 1),
               'predict_seconds_test_set': pred_s, 'n_test': len(te), 'parameters': int(m.count_params()),
               'rss_increase_mb': (psutil.Process().memory_info().rss - rss0) / 2**20}
        row.update(metrics(yte, (p > 0.5).astype(int), p)); DL_ROWS.append(row)
        DL_HIST[f'{proto}/{tag}/{rep}/{name}/{seed}'] = {k: [float(x) for x in v] for k, v in h.history.items() if k in ('loss', 'val_loss', 'accuracy', 'val_accuracy')}
        del m; tf.keras.backend.clear_session(); gc.collect(); return row

    t_dl = time.time()
    for proto in [p for p in PROTOCOLS if p in DL_PROTOCOLS]:
        runs = protocol_runs(proto, N_SEEDS_DL)
        for (tag, arg_), seed in zip(runs, itertools.cycle(DL_SEEDS)):
            tr, te = PROTOCOLS[proto](arg_)
            for rep in ('content', 'combined'):
                for name in DL_MODELS:
                    r = run_dl(proto, tag, arg_, seed, rep, name, tr, te)
                    print(f'[{time.time()-t_dl:7.0f}s] {proto}/{tag} {rep:9s} {name:17s} F1={r["f1"]:.4f} epochs={r["epochs_run"]} fit={r["fit_seconds"]:.0f}s')
            pd.DataFrame(DL_ROWS).to_csv(os.path.join(OUT_DIR, 'results_long_dl.csv'), index=False)
    save_json(DL_HIST, 'dl_training_histories.json')
else:
    print('Deep models skipped.')

In [ ]:
# ---------------------------------------------------------------- 9. STATISTICS
res_all = pd.concat([pd.DataFrame(RESULTS), pd.DataFrame(DL_ROWS)], ignore_index=True)
res_all.to_csv(os.path.join(OUT_DIR, 'results_long_all.csv'), index=False)

def ci95(x):
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return x.mean() - h, x.mean() + h

rows = []
for (proto, rep, mdl), gdf in res_all.groupby(['protocol', 'representation', 'model']):
    for met in ('f1', 'macro_f1', 'accuracy', 'auc', 'mcc'):
        v = gdf[met].values; lo, hi = ci95(v)
        rows.append({'protocol': proto, 'representation': rep, 'model': mdl, 'metric': met, 'n_runs': len(v), 'mean': v.mean(),
                     'sd': v.std(ddof=1) if len(v) > 1 else np.nan, 'ci95_low': lo, 'ci95_high': hi})
summary = pd.DataFrame(rows); summary.to_csv(os.path.join(OUT_DIR, 'summary_mean_sd_ci.csv'), index=False)

def nadeau_bengio(a, b, n_train, n_test):
    """corrected resampled paired t-test (Nadeau & Bengio, 2003) for repeated random sub-sampling"""
    d = np.asarray(a) - np.asarray(b); k = len(d)
    if k < 2 or d.std(ddof=1) == 0: return np.nan, np.nan
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * d.var(ddof=1)); return t, 2 * stats.t.sf(abs(t), k - 1)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); m = len(p); run = 0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(run, 1.0)
    return adj

tests = []
def compare(proto, a, b, metric='f1'):
    """a, b = (representation, model)"""
    ra = res_all[(res_all.protocol == proto) & (res_all.representation == a[0]) & (res_all.model == a[1])].sort_values('run')
    rb = res_all[(res_all.protocol == proto) & (res_all.representation == b[0]) & (res_all.model == b[1])].sort_values('run')
    k = min(len(ra), len(rb))
    if k < 2: return
    va, vb = ra[metric].values[:k], rb[metric].values[:k]
    tr_n = int(len(df) * (1 - TEST_FRAC)); te_n = int(len(df) * TEST_FRAC)
    t, p = nadeau_bengio(va, vb, tr_n, te_n)
    try: w = stats.wilcoxon(va, vb).pvalue
    except Exception: w = np.nan
    tests.append({'protocol': proto, 'A': f'{a[0]}/{a[1]}', 'B': f'{b[0]}/{b[1]}', 'metric': metric, 'n_pairs': k,
                  'mean_diff_A_minus_B': float((va - vb).mean()), 'nadeau_bengio_t': t, 'p_nadeau_bengio': p, 'p_wilcoxon': w})
for proto in PROTOCOLS:
    for m_ in ('LR', 'SVM'):
        compare(proto, ('combined', m_), ('content', m_))        # does adding the source help?
        compare(proto, ('content', m_), ('length', m_))          # does the text beat length/style alone?
    compare(proto, ('combined', 'LR'), ('source', 'SourceLookup'))
    for m_ in ('CNN', 'BiLSTM-Attention'):
        compare(proto, ('combined', m_), ('content', m_))
tests = pd.DataFrame(tests)
if len(tests):
    tests['p_holm'] = holm(tests.p_nadeau_bengio.fillna(1).values)
    tests.to_csv(os.path.join(OUT_DIR, 'significance_tests.csv'), index=False)
    print(tests.round(4).to_string())

# ---- figures for the paper: F1 per protocol, and confusion matrices with CORRECT labels (Real = 0, Fake = 1)
sel = [('none', 'Majority'), ('source', 'SourceLookup'), ('length', 'LR'), ('content', 'LR'), ('combined', 'LR'), ('content', 'CNN'), ('combined', 'CNN')]
protos = list(PROTOCOLS); w = 0.8 / len(sel)
fig, ax = plt.subplots(figsize=(10.5, 4.6))
for k, (rep, m_) in enumerate(sel):
    vals, errs = [], []
    for p_ in protos:
        r = summary[(summary.metric == 'f1') & (summary.protocol == p_) & (summary.representation == rep) & (summary.model == m_)]
        vals.append(r['mean'].iloc[0] if len(r) else np.nan); errs.append(np.nan_to_num(r['sd'].iloc[0]) if len(r) else 0)
    ax.bar(np.arange(len(protos)) + k * w, vals, w, yerr=errs, capsize=2, label=f'{rep} / {m_}')
ax.set_xticks(np.arange(len(protos)) + 0.4 - w / 2); ax.set_xticklabels(protos); ax.set_ylabel('F1 (Fake = positive), mean +- SD over runs')
ax.set_ylim(0.4, 1.02); ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig_f1_by_protocol.png'), dpi=300); plt.show()

cm_sel = [('combined', 'LR'), ('content', 'LR'), ('combined', 'CNN'), ('content', 'CNN')]
cm_sel = [c for c in cm_sel if ((res_all.representation == c[0]) & (res_all.model == c[1])).any()]
fig, axes = plt.subplots(2, len(cm_sel), figsize=(3.2 * len(cm_sel), 6.2), squeeze=False)
for ri, p_ in enumerate(('random', 'source_disjoint')):
    for ci, (rep, m_) in enumerate(cm_sel):
        r = res_all[(res_all.protocol == p_) & (res_all.representation == rep) & (res_all.model == m_) & (res_all.run == 'seed%d' % BASE_SEED)]
        a = axes[ri, ci]; a.axis('off')
        if not len(r): continue
        r = r.iloc[0]; cmx = np.array([[r.tn, r.fp], [r.fn, r.tp]], dtype=int); a.axis('on')
        a.imshow(cmx, cmap='Blues'); a.set_xticks([0, 1]); a.set_xticklabels(['Real', 'Fake']); a.set_yticks([0, 1]); a.set_yticklabels(['Real', 'Fake'])
        for i in range(2):
            for j in range(2): a.text(j, i, str(cmx[i, j]), ha='center', va='center', color='white' if cmx[i, j] > cmx.max() / 2 else 'black')
        a.set_title(f'{p_}\n{rep}/{m_}', fontsize=8); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig_confusion_matrices.png'), dpi=300); plt.show()

pv = summary[summary.metric == 'f1'].pivot_table(index=['representation', 'model'], columns='protocol', values='mean')
print('\nMean F1 (Fake = positive) by representation/model and protocol'); print(pv.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------- 10. EFFICIENCY (honest timings on the hardware in hardware.json)
EFF = []
tr, te = split_random(BASE_SEED); Xtr, Xte = FEATURES.iloc[tr], FEATURES.iloc[te]; ytr, yte = df.y.values[tr], df.y.values[te]
def single_doc_latency(predict_fn, X, n=200):
    ts = []
    for i in range(min(n, len(X))):
        t0 = time.perf_counter(); predict_fn(X.iloc[[i]]); ts.append(time.perf_counter() - t0)
    return float(np.median(ts) * 1000), float(np.percentile(ts, 95) * 1000)

for rep in ('combined', 'content'):
    for nm in ['LR', 'SVM', 'RF'] + (['XGB'] if HAVE_XGB else []):
        pipe = Pipeline([('prep', make_preprocessor(rep)), ('clf', base_models(BASE_SEED)[nm])])
        rss0 = psutil.Process().memory_info().rss
        t0 = time.time(); pipe.fit(Xtr, ytr); fit_s = time.time() - t0
        rss1 = psutil.Process().memory_info().rss
        t0 = time.time(); pipe.predict(Xte); batch_s = time.time() - t0
        med, p95 = single_doc_latency(pipe.predict, Xte)
        path = os.path.join(OUT_DIR, f'_tmp_{rep}_{nm}.joblib'); joblib.dump(pipe, path, compress=3); size_mb = os.path.getsize(path) / 2**20; os.remove(path)
        EFF.append({'representation': rep, 'model': nm, 'fit_seconds_fixed_hyperparameters': fit_s, 'batch_predict_seconds_incl_vectorisation': batch_s,
                    'docs_per_second_batch': len(Xte) / batch_s, 'single_doc_latency_ms_median': med, 'single_doc_latency_ms_p95': p95,
                    'model_size_mb_joblib_compressed': size_mb, 'rss_increase_during_fit_mb': (rss1 - rss0) / 2**20, 'n_test_docs': len(Xte)})
        print(EFF[-1])
if HAVE_TF and RUN_DL and DL_ROWS:
    d = pd.DataFrame(DL_ROWS); d = d[(d.protocol == 'random')]
    for (rep, nm), gg in d.groupby(['representation', 'model']):
        EFF.append({'representation': rep, 'model': nm, 'fit_seconds_fixed_hyperparameters': gg.fit_seconds.mean(),
                    'batch_predict_seconds_incl_vectorisation': gg.predict_seconds_test_set.mean(),
                    'docs_per_second_batch': gg.n_test.mean() / gg.predict_seconds_test_set.mean(), 'parameters': int(gg.parameters.iloc[0]),
                    'seconds_per_epoch': gg.seconds_per_epoch.mean(), 'epochs_run_mean': gg.epochs_run.mean(),
                    'rss_increase_during_fit_mb': gg.rss_increase_mb.mean(),
                    'note': 'tokenisation time not included; hardware: ' + ('GPU ' + str(HW.get('gpus_visible_to_tensorflow')) if HW.get('gpus_visible_to_tensorflow') else 'CPU only')})
pd.DataFrame(EFF).to_csv(os.path.join(OUT_DIR, 'efficiency.csv'), index=False)

In [ ]:
# ---------------------------------------------------------------- 11. CONVERGENCE EVIDENCE
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if DL_HIST:
    for key, h in DL_HIST.items():
        if not key.startswith('random/') or not key.endswith('/%d' % BASE_SEED): continue
        proto, tag, rep, name, seed = key.split('/')
        ls = '-' if rep == 'content' else '--'
        axes[0 if name == 'CNN' else 1].plot(h['loss'], ls, label=f'train {rep}'); axes[0 if name == 'CNN' else 1].plot(h['val_loss'], ls, label=f'val {rep}')
    for a, n in zip(axes, ('CNN', 'BiLSTM-Attention')):
        a.set_title(n + ' training / validation loss'); a.set_xlabel('epoch'); a.set_yscale('log'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig_dl_convergence.png'), dpi=300); plt.show()

# linear models: iterations needed; the earlier LinearSVC runs raised "Liblinear failed to converge"
conv = []
Xp = make_preprocessor('content').fit(Xtr); Xt = Xp.transform(Xtr)
for mi in (1000, 5000, 20000):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always'); s = LinearSVC(C=1.0, max_iter=mi, random_state=BASE_SEED).fit(Xt, ytr)
    conv.append({'model': 'LinearSVC', 'max_iter': mi, 'n_iter_used': int(np.max(s.n_iter_)), 'convergence_warning': any('converge' in str(x.message).lower() for x in w)})
l = LogisticRegression(C=1.0, solver='liblinear', max_iter=2000, random_state=BASE_SEED).fit(Xt, ytr)
conv.append({'model': 'LogisticRegression(liblinear)', 'max_iter': 2000, 'n_iter_used': int(np.max(l.n_iter_)), 'convergence_warning': False})
if HAVE_XGB:
    xs = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, tree_method='hist', eval_metric='logloss', n_jobs=-1, random_state=BASE_SEED)
    Xv = Xp.transform(Xte); xs.fit(Xt, ytr, eval_set=[(Xt, ytr), (Xv, yte)], verbose=False)
    ev = xs.evals_result(); plt.figure(figsize=(5, 4)); plt.plot(ev['validation_0']['logloss'], label='train'); plt.plot(ev['validation_1']['logloss'], label='test')
    plt.xlabel('boosting round'); plt.ylabel('log-loss'); plt.title('XGBoost (content-only) convergence'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'fig_xgb_convergence.png'), dpi=300); plt.show()
pd.DataFrame(conv).to_csv(os.path.join(OUT_DIR, 'convergence_linear_models.csv'), index=False); print(pd.DataFrame(conv))

In [ ]:
# ---------------------------------------------------------------- 12. EXPORT
import shutil
print('Files written to', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print('  ', f)
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print('\nDownload', OUT_DIR + '.zip', 'and send/paste the CSV/JSON files so the manuscript tables can be filled.')